# 04 - Forecast checks: the first live hours

The hourly forecast runs from go-live (2023-01-01), one run per hour, reading the simulated feed. This notebook checks the first replayed hours:

1. the forecasts: 77 per hour for each model in the registry, made only from what was known before the hour;
2. the data-quality gate and the run log;
3. how long a run takes, and what that means for replaying all eight live months.

In [1]:
import json
import os

import pandas as pd
import yaml

# run from the repo root, where main.py, config/ and datamart/ are
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

with open("config/pipeline.yaml") as f:
    config = yaml.safe_load(f)

# the tables are small, so pandas reads them directly
predictions = pd.read_parquet("datamart/gold/model_predictions")
data_quality = pd.read_parquet("datamart/gold/monitoring/data_quality")
run_log = pd.read_parquet("datamart/gold/monitoring/ops_run_log")
# the partition columns (date, run_id) come back as categories: use them as text
for df in (predictions, data_quality, run_log):
    df[["date", "run_id"]] = df[["date", "run_id"]].astype(str)
with open("model_bank/registry.json") as f:
    registry = json.load(f)
print("registry:", [(e["model_version"], e["status"]) for e in registry])

registry: [('B0_2021_12_20261014T050309', 'production'), ('M1_2021_12_20261014T050309', 'shadow')]


## 1 - Forecasts

Each hourly run for `[T−1, T)` writes the forecasts for hour T: one row per area for the production model, and one per area for the shadow. The planner view reads `role = production` only.

In [2]:
per_hour = predictions.groupby(["role", "model_version"]).agg(
    hours=("hour_start", "nunique"), rows=("hour_start", "size"),
    first_hour=("hour_start", "min"), last_hour=("hour_start", "max"))
print(per_hour)
print("rows per hour and role, always 77:", predictions.groupby(["hour_start", "role"]).size().unique())
print("rows where feature_cutoff != hour_start:", (predictions["feature_cutoff"] != predictions["hour_start"]).sum())
print("forecast times (UTC, the wall clock):", predictions["predicted_at"].min(), "to", predictions["predicted_at"].max())

                                       hours  rows          first_hour  \
role       model_version                                                 
production B0_2021_12_20261014T050309     49  3773 2023-01-01 01:00:00   
shadow     M1_2021_12_20261014T050309     49  3773 2023-01-01 01:00:00   

                                                last_hour  
role       model_version                                   
production B0_2021_12_20261014T050309 2023-01-03 01:00:00  
shadow     M1_2021_12_20261014T050309 2023-01-03 01:00:00  
rows per hour and role, always 77: [77]
rows where feature_cutoff != hour_start: 0
forecast times (UTC, the wall clock): 2026-10-14 06:23:11 to 2026-10-14 07:01:17


In [3]:
# the planner's question for the first 8-hour shift: where will the crashes be?
shift = predictions[(predictions["hour_start"] >= "2023-01-01 08:00") & (predictions["hour_start"] < "2023-01-01 16:00")]
expected = shift.groupby(["role", "community_area"])["predicted_count"].sum().round(2)
pd.concat({role: expected[role].nlargest(10).reset_index() for role in ["production", "shadow"]}, axis=1)

production                         shadow                
  community_area predicted_count community_area predicted_count
0             25            6.50             25            4.79
1             28            4.00             28            4.37
2              2            3.25              8            3.64
3              8            3.25             23            2.95
4             22            3.00             69            2.13
5             71            3.00             16            2.07
6              6            2.75             24            1.94
7             15            2.75              6            1.90
8             19            2.75              2            1.65
9             24            2.75             19            1.62

## 2 - Data quality and the run log

`check_data_quality` runs before inference and stops the run if a check fails. `log_run` writes one row per task of the run, whatever happened.

In [4]:
print(data_quality.groupby(["check", "status"]).agg(runs=("run_id", "nunique"), max_value=("value", "max")))
print()
print("runs logged:", run_log["run_id"].nunique(), "| tasks failed:", (run_log["status"] != "success").sum(),
      "| forecasts fresh:", run_log.groupby("run_id")["forecast_fresh"].first().value_counts().to_dict())
run_log.groupby("task")["seconds"].describe()[["count", "mean", "min", "max"]].round(1)

                                      runs  max_value
check                         status                 
cutoff_not_hour_start         ok        49       0.00
duplicate_keys                ok        49       0.00
rows                          ok        49      77.00
weather_reading_age_hours_max ok        49       0.15

runs logged: 49 | tasks failed: 0 | forecasts fresh: {True: 49}


,count,mean,min,max
task,,,,
bronze_crashes,49.0,9.9,7.7,13.0
bronze_weather,49.0,10.4,8.2,16.5
check_data_quality,49.0,1.3,0.6,2.4
gold_features,49.0,17.7,13.2,24.2
infer_production,49.0,1.5,0.9,2.5
infer_shadow,49.0,1.5,1.0,2.6
silver_crashes,49.0,13.5,10.6,18.0
silver_weather,49.0,12.9,9.2,15.9


## 3 - How long does an hour take?

Two numbers matter:
- **latency:** from the first task's start to the last one's end for one run. A forecast for T must exist before T, so this must stay well under an hour;
- **throughput:** during the replay many runs queue at once, and the `spark` pool (2 slots) is the bottleneck. Hours replayed per hour of wall clock decides whether all 5,832 live hours fit overnight.

In [5]:
log = run_log.assign(ended_at=run_log["started_at"] + pd.to_timedelta(run_log["seconds"], unit="s"))
per_run = log.groupby("run_id").agg(first_start=("started_at", "min"), last_end=("ended_at", "max"),
                                    task_seconds=("seconds", "sum"))
per_run["wall_seconds"] = (per_run["last_end"] - per_run["first_start"]).dt.total_seconds()
print(per_run[["task_seconds", "wall_seconds"]].describe().round(0).loc[["count", "mean", "min", "max"]])

# the replay of 1-2 January ran 48 hours at once; 3 January 00:00 ran alone afterwards
alone = per_run.loc["h_20230103T00"]
print(f"one run alone (h_20230103T00): {alone['wall_seconds']:.0f} s from first task to last, "
      f"{alone['task_seconds']:.0f} s inside the tasks")
replay = per_run[per_run.index < "h_20230103"]
span_hours = (replay["last_end"].max() - replay["first_start"].min()).total_seconds() / 3600
rate = len(replay) / span_hours
print(f"replay: {len(replay)} runs in {span_hours * 60:.0f} min -> {rate:.0f} hourly runs per hour of wall clock")
print(f"all 5,832 live hours at that rate: {5832 / rate:.0f} h")

       task_seconds  wall_seconds
count          49.0          49.0
mean           69.0          78.0
min            53.0          41.0
max            79.0          93.0
one run alone (h_20230103T00): 41 s from first task to last, 53 s inside the tasks
replay: 48 runs in 31 min -> 94 hourly runs per hour of wall clock
all 5,832 live hours at that rate: 62 h


**Findings:** (NOTE: Check these findings when rerunning)
- **Forecasts:** 49 replayed hours (1 January 01:00 to 3 January 01:00) × 77 areas, for each of the two models the registry names: **3,773 production (B0) and 3,773 shadow (M1) rows**, 77 per hour and role, `feature_cutoff = hour_start` in every row. `predicted_at` is the wall clock in UTC, which is why it says 2026.
- **The first shift:** both models put areas 25 and 28 first; below that their top 10s differ. That is what the shadow is for: the difference costs planners nothing until the promotion rule says M1 is better.
- **The gate and the log:** every data-quality check passed in all 49 runs, no task failed, and **every forecast was fresh**. The Spark tasks take 9–16 s each, mostly JVM start-up; the pandas tasks 1-2 s.
- **Run time:** one run on its own takes **41s**, well under the hour, so a live system would be on time. The replay ran **94 hourly runs per hour of wall clock**, so all 5,832 live hours would take **about 62h**: not overnight. The evidence points to a fallback: hourly runs for demo windows, and the rest of the period in day-sized intervals, through the same tasks.